## Push to GitHub — Provable Resilience Intelligence

**Prerequisites:**
1. GitHub repo `skhage/provable-resilience-intelligence` must exist (create via github.com/new or `gh repo create`)
2. Workspace Git credential (gitHubOAuth, ID 358347682222830) must have push access

**What this notebook does:**
1. Copies all project files to a temp git repo
2. Creates initial commit on `main`
3. Pushes to GitHub
4. Creates `develop` branch
5. Links the Databricks workspace to the repo

**Verified:** Databricks credential helper authenticates successfully to GitHub (tested 2026-10-20).

In [0]:
import subprocess, os, shutil, tempfile

# Project root in workspace
PROJECT_ROOT = "/Workspace/Users/stephen.hage@databricks.com/provable-resilience-intelligence"
GITHUB_REPO = "https://github.com/skhage/provable-resilience-intelligence.git"

# Create temp directory for git operations
tmpdir = tempfile.mkdtemp(prefix="pri-push-")
print(f"Working directory: {tmpdir}")

# Initialize git repo
for cmd in [
    ["git", "init"],
    ["git", "config", "user.email", "stephen.hage@databricks.com"],
    ["git", "config", "user.name", "skhage"],
    ["git", "branch", "-M", "main"],
]:
    subprocess.run(cmd, cwd=tmpdir, capture_output=True, check=True)

# Files and directories to copy (skip experiments, .databricks, __pycache__)
SKIP = {
    '.databricks', '__pycache__', '.ipynb_checkpoints', 
    'span_risk_classification', 'optical_anomaly_detection',  # MLflow experiments
    'resilience_agent_evaluation',  # MLflow experiment
    'devops_create_github_repo',  # Superseded notebook
    'devops_push_to_github',  # This notebook (meta)
}

def copy_project(src, dst):
    """Copy project files, converting .ipynb notebooks to .py where needed."""
    count = 0
    for item in os.listdir(src):
        if item in SKIP or item.startswith('.'):
            if item in ('.gitignore', '.assistant_instructions.md', '.github'):
                pass  # Include these dotfiles
            else:
                continue
        
        src_path = os.path.join(src, item)
        dst_path = os.path.join(dst, item)
        
        if os.path.isdir(src_path):
            os.makedirs(dst_path, exist_ok=True)
            count += copy_project(src_path, dst_path)
        else:
            shutil.copy2(src_path, dst_path)
            count += 1
    return count

file_count = copy_project(PROJECT_ROOT, tmpdir)
print(f"Copied {file_count} files to git staging area")

# List what we'll commit
result = subprocess.run(["find", ".", "-type", "f"], cwd=tmpdir, capture_output=True, text=True)
print(f"\nFiles to commit:\n{result.stdout}")

In [0]:
# Stage all files
subprocess.run(["git", "add", "."], cwd=tmpdir, check=True, capture_output=True)

# Commit
result = subprocess.run(
    ["git", "commit", "-m", "Initial commit: Provable Resilience Intelligence\n\nLakelink Fiber network resilience scoring platform.\n17 SID-aligned tables, 8 metric views, DAB config,\nReact scorecard app, ML models, CI/CD workflows.\n\nCo-authored-by: @pm <pm@agents>\nCo-authored-by: @data-engineer <de@agents>\nCo-authored-by: @data-analyst <da@agents>\nCo-authored-by: @ml-engineer <ml@agents>\nCo-authored-by: @app-developer <app@agents>\nCo-authored-by: @designer <ux@agents>\nCo-authored-by: @qa <qa@agents>\nCo-authored-by: @devops <devops@agents>"],
    cwd=tmpdir, capture_output=True, text=True
)
print(f"Commit: {result.stdout.strip()}")

# Set remote
subprocess.run(
    ["git", "remote", "add", "origin", GITHUB_REPO],
    cwd=tmpdir, capture_output=True
)

# Push main
print("\n--- Pushing to GitHub (main) ---")
env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
push = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    cwd=tmpdir, capture_output=True, text=True, timeout=60, env=env
)
print(f"Push rc: {push.returncode}")
if push.stdout: print(push.stdout)
if push.stderr: print(push.stderr)

if push.returncode == 0:
    print("\n✅ Successfully pushed to main!")
    
    # Create and push develop branch
    subprocess.run(["git", "checkout", "-b", "develop"], cwd=tmpdir, capture_output=True)
    dev_push = subprocess.run(
        ["git", "push", "-u", "origin", "develop"],
        cwd=tmpdir, capture_output=True, text=True, timeout=30, env=env
    )
    if dev_push.returncode == 0:
        print("✅ develop branch created and pushed!")
    else:
        print(f"⚠️ develop branch push failed: {dev_push.stderr}")
else:
    print("\n❌ Push failed. Ensure the GitHub repo exists at:")
    print(f"   {GITHUB_REPO}")
    print("   Create it at: https://github.com/new")

In [0]:
# After successful push, create a Databricks Repo (Git folder) linked to GitHub
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    repo = w.repos.create(
        url=GITHUB_REPO,
        provider="gitHub",
        path=f"/Repos/{w.current_user.me().user_name}/provable-resilience-intelligence"
    )
    print(f"✅ Databricks Repo created: {repo.path}")
    print(f"   Repo ID: {repo.id}")
    print(f"   Branch: {repo.branch}")
except Exception as e:
    print(f"⚠️ Repo creation: {e}")
    print("   You can create it manually: Workspace → Repos → Add Repo")